# Augmentation Analysis

This notebook visualizes preprocessing and augmentation steps used
for object detection training.

Goals:
- load the training split
- apply letterbox resize
- verify bounding box transformations
- inspect basic augmentations
- analyze bounding box sizes in the training dataset

In [ ]:
import json
import math
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt

from project_config import DETECTOR_TARGET_SIZE
from project_config import TRAIN_JSON, SEED
from utils.augmentations import (
    letterbox_resize,
    horizontal_flip,
    random_brightness,
)
from utils.visualization import draw_boxes

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print("Train samples:", len(train_data))

if not train_data:
    raise ValueError(f"Training split is empty: {TRAIN_JSON}")

## 1. Select a sample image

The first step is to load one sample from the training set and inspect:

- whether the image can be loaded correctly
- whether the corresponding bounding boxes exist
- the original image size
- the number of annotated objects

In [ ]:
sample = train_data[0]

image = cv2.imread(sample["image_path"])
if image is None:
    raise FileNotFoundError(f"Could not read image: {sample['image_path']}")

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
original_boxes = np.array(sample["boxes"], dtype=np.float32)

print("Sample file:", sample["filename"])
print("Objects:", len(original_boxes))
print("Original shape:", image.shape[:2])

## 2. Apply preprocessing and augmentations

The following steps are applied:

1. **letterbox resize** to a fixed square size
2. **horizontal flip**
3. **brightness adjustment**

The purpose is to verify that bounding boxes remain aligned with the objects
after each transformation.

In [ ]:
letterboxed_image, letterboxed_boxes, scale, pad_x, pad_y = letterbox_resize(
    image,
    original_boxes,
    target_size=960,
)

flipped_image, flipped_boxes = horizontal_flip(letterboxed_image, letterboxed_boxes)
bright_image = random_brightness(letterboxed_image)

## 3. Visual comparison of preprocessing steps

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

draw_boxes(axes[0, 0], image, original_boxes, title="Original image")
draw_boxes(axes[0, 1], letterboxed_image, letterboxed_boxes, title="Letterbox resize")
draw_boxes(axes[1, 0], flipped_image, flipped_boxes, title="Horizontal flip")
draw_boxes(axes[1, 1], bright_image, letterboxed_boxes, title="Brightness augmentation")

plt.tight_layout()
plt.show()

## 4. Figure for thesis documentation

The following figure can be used as an illustrative example in the thesis.
It demonstrates:

- aspect ratio preservation during resize
- correct transformation of bounding boxes
- the effect of simple augmentations on the input image

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

draw_boxes(axes[0, 0], image, original_boxes, title="Original image")
draw_boxes(axes[0, 1], letterboxed_image, letterboxed_boxes, title="Letterbox resize")
draw_boxes(axes[1, 0], flipped_image, flipped_boxes, title="Horizontal flip")
draw_boxes(axes[1, 1], bright_image, letterboxed_boxes, title="Brightness augmentation")

plt.tight_layout()
plt.show()

## 5. Visual dataset inspection

This step displays multiple random training samples with their bounding boxes.
The goal is to verify that:

- images load correctly
- annotations visually match the objects
- the dataset contains a reasonable range of object counts and image sizes

In [ ]:
def show_random_samples_with_boxes(dataset, num_samples=20, seed=SEED):
    random.seed(seed)

    selected = random.sample(dataset, min(num_samples, len(dataset)))
    cols = 4
    rows = math.ceil(len(selected) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, sample in zip(axes, selected):
        image = cv2.imread(sample["image_path"])
        if image is None:
            ax.set_title(f"Could not read\n{sample.get('filename', 'unknown')}")
            ax.axis("off")
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        boxes = np.array(sample["boxes"], dtype=np.float32)

        h, w = image.shape[:2]
        title = f"{sample.get('filename', 'unknown')}\nshape={w}x{h}, boxes={len(boxes)}"
        draw_boxes(ax, image, boxes, title=title, fontsize=9)

    for ax in axes[len(selected):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 6. Bounding box size analysis

This step computes basic statistics of bounding boxes in the training dataset.

The analysis focuses on:

- bounding box widths and heights
- bounding box areas
- relative object size compared to image area
- the proportion of very small objects after resizing

These statistics help estimate the difficulty of the detection task.

In [ ]:
def collect_box_stats(dataset, target_size=960):
    orig_widths = []
    orig_heights = []
    orig_areas = []
    orig_area_ratios = []

    resized_widths = []
    resized_heights = []
    resized_areas = []
    resized_area_ratios = []

    invalid_boxes = []
    missing_images = []

    for i, sample in enumerate(dataset):
        image = cv2.imread(sample["image_path"])
        if image is None:
            missing_images.append(sample.get("filename", f"index_{i}"))
            continue

        h, w = image.shape[:2]
        boxes = np.array(sample["boxes"], dtype=np.float32)

        if len(boxes) == 0:
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        _, resized_boxes, _, _, _ = letterbox_resize(
            image_rgb,
            boxes,
            target_size=target_size,
        )

        for j, (xmin, ymin, xmax, ymax) in enumerate(boxes):
            bw = xmax - xmin
            bh = ymax - ymin
            area = bw * bh

            if bw <= 0 or bh <= 0:
                invalid_boxes.append({
                    "sample": sample.get("filename", f"index_{i}"),
                    "box_index": j,
                    "box": [float(xmin), float(ymin), float(xmax), float(ymax)],
                })
                continue

            orig_widths.append(bw)
            orig_heights.append(bh)
            orig_areas.append(area)
            orig_area_ratios.append(area / (w * h))

        for xmin, ymin, xmax, ymax in resized_boxes:
            bw = xmax - xmin
            bh = ymax - ymin
            area = bw * bh

            if bw <= 0 or bh <= 0:
                continue

            resized_widths.append(bw)
            resized_heights.append(bh)
            resized_areas.append(area)
            resized_area_ratios.append(area / (target_size * target_size))

    def summarize(name, values):
        values = np.array(values, dtype=np.float32)
        if len(values) == 0:
            print(f"{name}: no data")
            return

        print(f"{name}:")
        print(f"  count   = {len(values)}")
        print(f"  min     = {values.min():.2f}")
        print(f"  p10     = {np.percentile(values, 10):.2f}")
        print(f"  median  = {np.median(values):.2f}")
        print(f"  mean    = {values.mean():.2f}")
        print(f"  p90     = {np.percentile(values, 90):.2f}")
        print(f"  max     = {values.max():.2f}")
        print()

    print("=" * 60)
    print("BOUNDING BOX STATISTICS")
    print("=" * 60)

    summarize("Original box width [px]", orig_widths)
    summarize("Original box height [px]", orig_heights)
    summarize("Original box area [px^2]", orig_areas)
    summarize("Original box area ratio", orig_area_ratios)

    summarize(f"Resized box width after letterbox to {target_size} [px]", resized_widths)
    summarize(f"Resized box height after letterbox to {target_size} [px]", resized_heights)
    summarize(f"Resized box area after letterbox to {target_size} [px^2]", resized_areas)
    summarize("Resized box area ratio", resized_area_ratios)

    print("=" * 60)
    print(f"Invalid boxes: {len(invalid_boxes)}")
    if invalid_boxes[:5]:
        print("Examples of invalid boxes:")
        for item in invalid_boxes[:5]:
            print(item)

    print()
    print(f"Missing images: {len(missing_images)}")
    if missing_images[:5]:
        print("Examples of missing files:")
        for item in missing_images[:5]:
            print(item)

    if len(resized_widths) > 0 and len(resized_heights) > 0:
        resized_widths_np = np.array(resized_widths)
        resized_heights_np = np.array(resized_heights)

        small_8 = np.mean((resized_widths_np < 8) | (resized_heights_np < 8)) * 100
        small_16 = np.mean((resized_widths_np < 16) | (resized_heights_np < 16)) * 100
        small_24 = np.mean((resized_widths_np < 24) | (resized_heights_np < 24)) * 100

        print()
        print("SMALL OBJECT RATIO AFTER RESIZE:")
        print(f"  box width or height < 8 px  : {small_8:.2f}%")
        print(f"  box width or height < 16 px : {small_16:.2f}%")
        print(f"  box width or height < 24 px : {small_24:.2f}%")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].hist(orig_widths, bins=30)
    axes[0, 0].set_title("Original box widths")

    axes[0, 1].hist(orig_heights, bins=30)
    axes[0, 1].set_title("Original box heights")

    axes[1, 0].hist(resized_widths, bins=30)
    axes[1, 0].set_title(f"Resized box widths ({target_size})")

    axes[1, 1].hist(resized_heights, bins=30)
    axes[1, 1].set_title(f"Resized box heights ({target_size})")

    plt.tight_layout()
    plt.show()

In [ ]:
show_random_samples_with_boxes(train_data, num_samples=20, seed=SEED)
collect_box_stats(train_data, target_size=DETECTOR_TARGET_SIZE)